## Operational Flow 
__In the notebook we will design the complete operation flow from simple text SQL query to reasoning on related AST instance graph and reconstructing updated SQL text query back based on the reasoning results.__



In [26]:
query = "SELECT bank, SUM(amount), type AS total FROM Transaction WHERE (account_id = 123) and (amount > 100) GROUP BY account_id"
# query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

# query = """
# SELECT
#     c.customer_id,
#     c.name,
#     (
#         SELECT COUNT(*)
#         FROM orders o
#         WHERE o.customer_id = c.customer_id
#     ) AS total_orders,
#     (
#         SELECT AVG(o2.total_amount)
#         FROM orders o2
#         WHERE o2.customer_id = c.customer_id
#           AND o2.order_date > CURRENT_DATE - INTERVAL '30 days'
#     ) AS avg_recent_order_amount
# FROM customers c
# WHERE c.status = 'active';

# """

# query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

### Step-1 : SQL to AST conversion

In [27]:
from sqlglot import parse_one, exp

In [28]:
ast_obj = parse_one(query)

### Step-2 : AST (`sqlgpt_parser`) to AST Tree conversion

In [29]:
import uuid

In [ ]:
# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause",
    "Order": "OrderByClause",
    "Having": "HavingClause",
    "Join": "JoinClause"
}

STATEMENT_TYPES = {
    "Delete": "DeleteStatement",
    "Insert": "InsertStatement",
    "Update": "UpdateStatement",
    "Select": "SelectStatement"
}

EXPRESSION_CATEGORIES = {
    "Alias": "Alias",
    "Table": "TableRef",
    "Column": "ColumnRef",
    "Wildcard": "WildCard",
    "Identifier": "Identifier",
    # Functions
    "Sum": "Function",
    "Count": "Function",
    "Avg": "Function",
    "Max": "Function",
    "Min": "Function",
    "Concat": "Function",
    "Coalesce": "Function",
    # Operators
    "And": "Operator",
    "Or": "Operator",
    "EQ": "Operator",
    "GT": "Operator",
    "LT": "Operator",
    "Add": "Operator",
    "Sub": "Operator",
    "Mul": "Operator",
    "Div": "Operator",
    # Literals
    "Literal": "Literal"
} # TODO: Add more expression types as needed

def map_node_type(node, statement=False): # TODO: look the logic is complete but needs to be tested
    """
    Map a sqlglot AST node to its corresponding type in the ontology.
    """
    class_name = type(node).__name__
    if statement:
        if class_name in STATEMENT_TYPES:
            return "Statement", STATEMENT_TYPES[class_name]
        else:
            return "Statement", class_name
    else:
        if class_name in CLAUSE_TYPES:
            return "Clause", CLAUSE_TYPES[class_name]
        elif class_name in EXPRESSION_CATEGORIES: # TODO: I need to set the specific category for each expression type in ast object
            return "Expression", EXPRESSION_CATEGORIES[class_name]
        else:
            return "Expression", class_name


In [31]:

# --- TreeNode Class ---

class TreeNode:
    """ 
    TreeNode represents a node in the SQL AST tree. It can be a statement, clause, or expression.
    """
    def __init__(self, sqlglot_node, parent=None):
        self.remove = False
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node, statement=True if parent is None else False)

        self.id = "n"+str(uuid.uuid4())[:6]
        sqlglot_node.meta['id'] = self.id

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            # self.reference_id = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable = parts[0].this
                self.refcol = parts[1].this
            elif len(parts) == 1:
                self.refcol = parts[0].this

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name
            if sqlglot_node.alias:
                self.refalias = sqlglot_node.alias
            # self.reference_id = None

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

        if self.name == "Literal":
            self.value = sqlglot_node.this
        
        if self.name == "Operator":
            if (type(sqlglot_node).__name__ in ["And", "Or"]):
                self.logics = True 
            else:
                self.logics = False

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        if self.name == "Literal" and hasattr(self, "value"):
            suffix += f" [value={self.value}]"

        if hasattr(self, "reference_id"):
            suffix += f" [reference_id={self.reference_id}]"
        if hasattr(self, "remove"):
            suffix += f" [remove={self.remove}]"
        if hasattr(self, "refalias"):
            suffix += f" [refalias={self.refalias}]"
        if hasattr(self, "logics"):
            suffix += f" [logics={self.logics}]"
        return f"<({self.id}) {self.kind} | {suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)


In [32]:
# --- Tree Builder ---

def build_tree(sqlglot_node, parent=None):

    if parent is not None:
        clause_root = TreeNode(sqlglot_node, parent=parent)
        root_node = parent
        root_node.add_child(clause_root)
    else:
        root_node = TreeNode(sqlglot_node)
        clause_root = TreeNode(sqlglot_node, parent=root_node)
        root_node.add_child(clause_root)
    

    found_first_clause = False
    for key, child in sqlglot_node.args.items():
        if child is None:
            continue

        children = child if isinstance(child, list) else [child]

        for expr in children:
            if not isinstance(expr, exp.Expression): # did not understand the logic of the check
                continue

            c_kind, c_name = map_node_type(expr)

            if not found_first_clause and c_kind != "Clause":
                sub = _build_recursive(expr, clause_root)
                if sub:
                    clause_root.add_child(sub)
            else:
                found_first_clause = True
                clause_node = TreeNode(expr, parent=root_node)
                root_node.add_child(clause_node)

                for _, grandchild in expr.args.items():
                    if isinstance(grandchild, exp.Expression):
                        if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier): # this is a weak check, so try to transfer this is recursion
                            continue  # Skip Identifier under Table
                        child_node = _build_recursive(grandchild, clause_node)
                        if child_node:
                            clause_node.add_child(child_node)
                    elif isinstance(grandchild, list):
                        for item in grandchild:
                            if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                continue  # Skip Identifier under Table
                            if isinstance(item, exp.Expression):
                                child_node = _build_recursive(item, clause_node)
                                if child_node:
                                    clause_node.add_child(child_node)
    if parent is None:
        return root_node


def _build_recursive(sqlglot_node, parent_node):
    if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "ColumnRef":
        return None

    if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "TableRef":
        return None
    
    if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "Alias":
        return None

    if isinstance(sqlglot_node, exp.TableAlias) and parent_node.name == "TableRef":
        return None

    if isinstance(sqlglot_node, exp.Paren):
        # return first child of the paren expression to avoid unnecessary nesting
        first_child = next(iter(sqlglot_node.args.values()))
        return _build_recursive(first_child, parent_node)

    if isinstance(sqlglot_node, exp.Select):
        current = build_tree(sqlglot_node, parent_node)
    else:
        current = TreeNode(sqlglot_node, parent_node)
        for _, child in sqlglot_node.args.items():
            if isinstance(child, exp.Expression):
                if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                    continue
                child_node = _build_recursive(child, current)
                if child_node:
                    current.add_child(child_node)
            elif isinstance(child, list):
                for item in child:
                    if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                        continue
                    if isinstance(item, exp.Expression):
                        child_node = _build_recursive(item, current)
                        if child_node:
                            current.add_child(child_node)

    return current


In [33]:
ast = parse_one(query)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()

<(n3a414c) Statement |  (SelectStatement) [remove=False]>
  <(n1187d9) Clause |  (SelectClause) [remove=False]>
    <(n27a982) Expression |  (ColumnRef) [reftable=None, refcol=bank] [remove=False]>
    <(n34b434) Expression |  (Function) [remove=False]>
      <(nddc9f2) Expression |  (ColumnRef) [reftable=None, refcol=amount] [remove=False]>
    <(n95693e) Expression |  (Alias) [name=total] [remove=False]>
      <(n99f27d) Expression |  (ColumnRef) [reftable=None, refcol=type] [remove=False]>
  <(nc314ed) Clause |  (FromClause) [remove=False]>
    <(n271912) Expression |  (TableRef) [reftable=Transaction] [remove=False]>
  <(n812e14) Clause |  (WhereClause) [remove=False]>
    <(n9c03e9) Expression |  (Operator) [remove=False] [logics=True]>
      <(n8a958d) Expression |  (Operator) [remove=False] [logics=False]>
        <(nedbaac) Expression |  (ColumnRef) [reftable=None, refcol=account_id] [remove=False]>
        <(n88fa20) Expression |  (Literal) [value=123] [remove=False]>
      <(

In [34]:
def get_node_children(node):
    """Helper to get all child expression nodes from a node's args."""
    for value in node.args.values():
        if isinstance(value, exp.Expression):
            yield value
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, exp.Expression):
                    yield item

def pretty_print_ast(node, prefix="", is_last=True):
    """
    Recursively prints a formatted tree of the AST, including our custom IDs.
    """
    # Use the first 8 characters of the UUID for cleaner printing
    short_id = node.meta.get('id', 'N/A')[:8]
    
    # Get the node's type and any direct value (like a column name)
    node_name = node.__class__.__name__
    if hasattr(node, 'this'):
        node_name += f" ({node.this})"

    # Print the current node with a tree-like structure
    connector = "└── " if is_last else "├── "
    print(f"{prefix}{connector}{node_name}  [ID: {short_id}]")

    # Recurse for children
    children = list(get_node_children(node))
    for i, child in enumerate(children):
        new_prefix = prefix + ("    " if is_last else "│   ")
        pretty_print_ast(child, prefix=new_prefix, is_last=(i == len(children) - 1))

In [35]:
pretty_print_ast(ast)

└── Select (None)  [ID: n1187d9]
    ├── Column (bank)  [ID: n27a982]
    │   └── Identifier (bank)  [ID: N/A]
    ├── Sum (amount)  [ID: n34b434]
    │   └── Column (amount)  [ID: nddc9f2]
    │       └── Identifier (amount)  [ID: N/A]
    ├── Alias (type)  [ID: n95693e]
    │   ├── Column (type)  [ID: n99f27d]
    │   │   └── Identifier (type)  [ID: N/A]
    │   └── Identifier (total)  [ID: N/A]
    ├── From (Transaction)  [ID: nc314ed]
    │   └── Table (Transaction)  [ID: n271912]
    │       └── Identifier (Transaction)  [ID: N/A]
    ├── Where ((account_id = 123) AND (amount > 100))  [ID: n812e14]
    │   └── And ((account_id = 123))  [ID: n9c03e9]
    │       ├── Paren (account_id = 123)  [ID: N/A]
    │       │   └── EQ (account_id)  [ID: n8a958d]
    │       │       ├── Column (account_id)  [ID: nedbaac]
    │       │       │   └── Identifier (account_id)  [ID: N/A]
    │       │       └── Literal (123)  [ID: n88fa20]
    │       └── Paren (amount > 100)  [ID: N/A]
    │      

### Step-3 : AST Tree to Ontology Instance 

In [9]:
# load ontology
from owlready2 import get_ontology, sync_reasoner
import uuid

onto_path = "../ontology_file/aput.rdf"
onto = get_ontology(onto_path).load()

In [10]:
for t in onto.individuals():
    s = [cls.name for cls in t.is_a if type(cls).__name__ == "ThingClass"]
    print(s)
    if "Column" in s:
        print(t.name)

    # for c in t.is_a:
        # print(type(c).__name__, c.name if type(c).__name__ == "ThingClass" else "None")
        # if len(t.is_a) == 2 and "Table" == t.is_a[1].name:
        # if "Table" in [cls.name for cls in t.is_a]:
        #     table_entities[t.TableName[0]] = t.name

['AccessAlignmentState']
['AccessAlignmentState']
['GrantLevel']
['GrantType']
['GrantLevel']
['Action']
['ActionScope']
['ActionScope']
['Action']
['ActionScope']
['PolicyCorrectnessStatus']
['Action']
['GrantType']
['GrantLevel']
['GrantType']
['ActionScope', 'SQLNode']
['Action']
['AccessAlignmentState']
['Action']
['ActionScope']
['ActionScope', 'SQLNode', 'Statement']
['PolicyCorrectnessStatus']
['Agent']
['Column', 'Object']
c001
['Object', 'Table']
['Column', 'Object']
c002
['Object', 'Table']
['Object', 'Table']
['Object', 'Table']
['Object', 'Table']
['Column', 'Object']
c003
['Column', 'Object']
c004
['Column', 'Object']
c005
['Column', 'Object']
c006
['Column', 'Object']
c007
['Column', 'Object']
c008
['Column', 'Object']
c009
['Column', 'Object']
c010
['Column', 'Object']
c011
['Column', 'Object']
c012
['Column', 'Object']
c013
['Column', 'Object']
c014
['Object', 'Table']
['Object', 'Table']
['Column', 'Object']
c015
['Column', 'Object']
c016
['Column', 'Object']
c017
['Co

In [11]:
def resolve_references_with_ontology(root_node, ontology):

    table_entities = {}
    for t in ontology.individuals():
        # if len(t.is_a) == 2 and "Table" == t.is_a[1].name:
        if "Table" in [cls.name for cls in t.is_a if type(cls).__name__ == "ThingClass"]:
            table_entities[t.TableName[0]] = t.name
    # print(table_entities)
    
    column_lookup = {}
    for n in ontology.individuals():
        # if "Column" == n.is_a[0].name:
        if "Column" in [cls.name for cls in n.is_a if type(cls).__name__ == "ThingClass"]:
            # print("Here")
            for tbl in getattr(n, "columnOfTable", []):
                table_name = tbl.TableName[0]
                column_lookup[(table_name, n.ColumnName[0])] = n.name
    # print(column_lookup)

    def resolve_for_statement(stmt_node):
        alias_map = {}  # alias -> table name
        tables_in_scope = []  # actual tables in FROM clause

        for child in stmt_node.children:
            if child.name == "FromClause":
                for t in child.walk():
                    if t.name == "TableRef" and hasattr(t, "reftable"):
                        table_name = t.reftable
                        alias_map[table_name] = table_name
                        tables_in_scope.append(table_name)

                        if hasattr(t, "refAlias"):
                            alias_map[t.refAlias] = table_name

        for node in stmt_node.walk():
            if node.name == "TableRef" and hasattr(node, "reftable"):
                tbl_key = node.reftable
                ref_table = table_entities.get(tbl_key)
                if ref_table:
                    node.reference_id = table_entities[node.reftable]

            elif node.name == "ColumnRef" and hasattr(node, "refcol"):
                # Use declared reftable if exists
                if hasattr(node, "reftable") and node.reftable:
                    tbl_key = alias_map.get(node.reftable, node.reftable)
                    col_key = (tbl_key, node.refcol)
                else:
                    if len(tables_in_scope) == 1:
                        tbl_key = tables_in_scope[0]
                        # print(f"[Info] Using single table '{tbl_key}' for column '{node.refcol}'.")
                        col_key = (tbl_key, node.refcol)
                    else:
                        print(f"[Warning] Ambiguous column '{node.refcol}' with no reftable in multi-table query.")
                        continue
                

                ref_col = column_lookup.get(col_key)
                # print(f"[Debug] Resolving ColumnRef {ref_col} with key {col_key}.")
                if ref_col:
                    node.reference_id = ref_col

    for node in root_node.walk():
        if node.kind == "Statement":
            resolve_for_statement(node)


In [12]:
resolve_references_with_ontology(tree_root, onto)
tree_root.print_tree()

<(n14d05c) Statement |  (SelectStatement) [remove=False]>
  <(n93e631) Clause |  (SelectClause) [remove=False]>
    <(nc52c38) Expression |  (ColumnRef) [reftable=None, refcol=bank] [reference_id=c012] [remove=False]>
    <(n2e03b4) Expression |  (Function) [remove=False]>
      <(nb0b380) Expression |  (ColumnRef) [reftable=None, refcol=amount] [reference_id=c031] [remove=False]>
    <(nadcb37) Expression |  (Alias) [name=total] [remove=False]>
      <(n43127e) Expression |  (ColumnRef) [reftable=None, refcol=type] [reference_id=c009] [remove=False]>
  <(nae726a) Clause |  (FromClause) [remove=False]>
    <(n55d218) Expression |  (TableRef) [reftable=Transaction] [reference_id=t002] [remove=False]>
  <(na934a5) Clause |  (WhereClause) [remove=False]>
    <(n874752) Expression |  (Operator) [remove=False] [logics=True]>
      <(n8a803a) Expression |  (Operator) [remove=False] [logics=False]>
        <(n64d579) Expression |  (ColumnRef) [reftable=None, refcol=account_id] [reference_id=c

### Ontology reasoning

In [ ]:
class OntologyInstantiator:
    def __init__(self, onto_path):
        """
        Initializes the instantiator by loading the ontology.
        """
        self.onto = get_ontology(onto_path).load()
        self.class_lookup = {cls.name: cls for cls in self.onto.classes()}
        
        # Tracking lists are now instance attributes
        self.table_ref_instances = []
        self.column_ref_instances = []

    def get_table_ref_instances(self):
        """
        Returns the list of instantiated TableRef objects.
        """
        return self.table_ref_instances
    
    def get_column_ref_instances(self):
        """
        Returns the list of instantiated ColumnRef objects.
        """
        return self.column_ref_instances

    def walk_and_instantiate(self, node, agent_id, parent_instance=None):
        """
        Recursively walks a tree and instantiates ontology classes.
        All dependencies like 'self.onto' and 'self.class_lookup' are explicit.
        """
        node_class = self.class_lookup.get(node.name)
        if not node_class:
            print(f"[Warning] Missing class in ontology: {node.name}")
            return None

        inst = node_class(node.id)

        # Attach parent link
        if parent_instance:
            inst.immediateParentNode.append(parent_instance)

        if STATEMENT_TYPES.get(node.name, None):
            ref_obj = self.onto.search_one(iri="*"+agent_id)
            if ref_obj:
                inst.executedBy.append(ref_obj)

        # Link TableRef to ontology Table via reference_id
        if node.name == "TableRef":
            self.table_ref_instances.append(inst)
            if hasattr(node, "reference_id"):
                ref_obj = self.onto.search_one(iri="*" + node.reference_id)
                if ref_obj:
                    inst.referencesToTable.append(ref_obj)
                else:
                    print(f"[Warning] TableRef unresolved: {node.reference_id}")

        # Link ColumnRef to ontology Column via reference_id
        if node.name == "ColumnRef":
            self.column_ref_instances.append(inst)
            if hasattr(node, "reference_id"):
                ref_obj = self.onto.search_one(iri="*" + node.reference_id)
                if ref_obj:
                    inst.referencesToColumn.append(ref_obj)
                else:
                    print(f"[Warning] ColumnRef unresolved: {node.reference_id}")

        # Recurse
        for child in node.children:
            self.walk_and_instantiate(child, agent_id, parent_instance=inst)

        return inst

    def process_tree(self, tree_root, agent_id):
        """
        Main method to process the tree, run the reasoner, and save the result.
        """
        with self.onto:
            # Assume resolve_references_with_ontology is defined elsewhere
            # and now takes the ontology as an argument if needed.
            # resolve_references_with_ontology(tree_root, self.onto) 
            
            self.walk_and_instantiate(tree_root, agent_id)
            sync_reasoner(infer_property_values=True, debug=0)
    
    def save(self, output_path, file_format="rdfxml"):
        """
        Saves the modified ontology to a file.
        """
        self.onto.save(file=output_path, format=file_format)

In [21]:
onto_path = "../ontology_file/aput.rdf"
output_path = "../ontology_file/resolved_query_instances.owl"
agent_id = "a0012"

# 1. Create an instance of the class
instantiator = OntologyInstantiator(onto_path)

# 2. Process the tree
instantiator.process_tree(tree_root, agent_id=agent_id)

# 3. Save the results
instantiator.save(output_path)

print("Ontology processing complete and saved.")


Ontology processing complete and saved.


In [23]:
table_ref_instances = instantiator.get_table_ref_instances()
column_ref_instances = instantiator.get_column_ref_instances()

for inst in table_ref_instances + column_ref_instances:
    print(f"{inst.name} → Alignment: {getattr(inst, 'hasAlignmentState', 'N/A')}")


n55d218 → Alignment: [aput.Aligned]
nc52c38 → Alignment: []
nb0b380 → Alignment: []
n43127e → Alignment: [aput.Violated]
n64d579 → Alignment: [aput.Violated]
n7f5ab4 → Alignment: []
nb97d35 → Alignment: [aput.Violated]


### Ontology to AST

In [24]:
column_ref_dict = {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in column_ref_instances }

table_ref_dict = {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in table_ref_instances}

In [25]:
column_ref_dict

{'nc52c38': 'Aligned',
 'nb0b380': 'Aligned',
 'n43127e': 'Violated',
 'n64d579': 'Violated',
 'n7f5ab4': 'Aligned',
 'nb97d35': 'Violated'}

In [16]:
def soft_prune_ast(root, column_ref_instances, table_ref_instances):
    """
    Traverse TreeNode tree and soft-prune by setting `remove = True` on nodes based on alignment states.
    """

    for node in root.walk():
        # Handle ColumnRef node logic
        if node.name == "ColumnRef":
            state = column_ref_instances.get(node.id)
            if state == "Violated":
                print(f"ColumnRef node {node.id} has alignment state Violated.")
                parent = node.parent
                print(f"Parent of ColumnRef node {node.id} is {parent.name if parent else 'None'}.")
                if parent:
                    if parent.name == "SelectClause":
                        node.remove = True
                        print(f"Marking ColumnRef node {node.id} for removal (in Clause).")
                    elif parent.name in ("Operator", "Function"):
                        parent.remove = True
                        print(f"Marking parent {parent.id} of ColumnRef node {node.id} for removal (in Operator/Function).")

        # Handle TableRef node logic
        elif node.name == "TableRef":
            state= table_ref_instances.get(node.id)
            if state == "Violated":
                # Find immediate statement node
                parent = node.parent
                while parent and parent.kind != "Statement":
                    parent = parent.parent
                if parent:
                    parent.remove = True
                    print(f"Marking Statement node {parent.id} for removal due to TableRef {node.id}.")


In [17]:
column_ref_instances

[aput.nb4aea1, aput.na19eab, aput.nc1f28c, aput.n3f6a62]

In [18]:
soft_prune_ast(tree_root, column_ref_dict, table_ref_dict)

# To inspect:
tree_root.print_tree()



ColumnRef node n3f6a62 has alignment state Violated.
Parent of ColumnRef node n3f6a62 is SelectClause.
Marking ColumnRef node n3f6a62 for removal (in Clause).
<(qf9e7a1) Statement |  (SelectStatement) [remove=False]>
  <(cf9e7a1) Clause |  (SelectClause) [remove=False]>
    <(nb4aea1) Expression |  (ColumnRef) [reftable=None, refcol=account_id] [reference_id=c002] [remove=False]>
    <(na19eab) Expression |  (ColumnRef) [reftable=None, refcol=account_to] [reference_id=c004] [remove=False]>
    <(nc1f28c) Expression |  (ColumnRef) [reftable=None, refcol=bank_to] [reference_id=c003] [remove=False]>
    <(n3f6a62) Expression |  (ColumnRef) [reftable=None, refcol=amount] [reference_id=c005] [remove=True]>
  <(n9bcca2) Clause |  (FromClause) [remove=False]>
    <(n772064) Expression |  (TableRef) [reftable=Order] [reference_id=t001] [remove=False]>


In [ ]:
def recursive_prune(node):
    """ 
    Recursively prune the tree by setting `remove = True` on nodes that are marked for removal.
    """

    for child in node.children:
        recursive_prune(child)
    
    if node.name in ["ColumnRef", "TableRef"]:
        state = column_ref_dict.get(node.id, "Aligned")
        if state == "Violated":
            node.remove = True 

    if node.name in EXPRESSION_CATEGORIES.values():
        if any(child.remove for child in node.children):
            if node.logics:
                # get non-removed children 
                non_removed_children = [child for child in node.children if not child.remove]
                node.parent.add_child(non_removed_children)
            
            node.remove = True
        


In [ ]:
import uuid
from sqlglot import parse_one, exp

class ASTPruner:
    """
    A robust class to parse a SQL query and prune it based on a set of
    invalid node IDs, working directly with the sqlglot AST.
    """
    def __init__(self, sql: str, violated_ids: set):
        self.violated_ids = violated_ids
        
        # Parse and decorate the AST upon initialization
        ast = parse_one(sql)
        self.decorated_ast = self._decorate_ast_with_ids(ast)

    def _decorate_ast_with_ids(self, ast_node):
        """Adds a unique ID to a 'meta' attribute on each node."""
        for node in ast_node.walk():
            if not hasattr(node, 'meta'):
                node.meta = {}
            node.meta['id'] = str(uuid.uuid4())
        return ast_node

    def _pruning_transformer(self, node):
        """
        The core logic passed to ast.transform(). It prunes the tree
        bottom-up based on the user's rules.
        """
        # Base Case: If the node itself is explicitly violated, remove it.
        if hasattr(node, 'meta') and node.meta.get('id') in self.violated_ids:
            print(f"--> Removing violated node {node.this if hasattr(node, 'this') else node.__class__.__name__} [ID: {node.meta['id']}]")
            return None # Returning None deletes the node

        # Recursive Cases: Check if a parent node has become invalid because
        # its essential children were removed by previous transformations.

        # Rule: If a Function's argument is removed, remove the function.
        # (e.g., SUM(invalid_column) -> SUM(None) -> remove SUM)
        if isinstance(node, exp.Func) and node.this is None:
            print(f"--> Cascading removal of Function '{node.sql()}' because its argument was removed.")
            return None

        # Rule: If a binary Operator's operands are removed, remove the operator.
        # (e.g., invalid_column = 5 -> None = 5 -> remove =)
        if isinstance(node, exp.Binary) and (node.left is None or node.right is None):
            print(f"--> Cascading removal of Operator '{node.sql()}' because an operand was removed.")
            return None

        # Rule: If a JOIN's 'ON' clause becomes invalid/empty, remove the JOIN.
        if isinstance(node, exp.Join) and node.on is None:
            print(f"--> Cascading removal of JOIN '{node.sql()}' because its ON clause was removed.")
            return None

        # Final check: For a SELECT statement, if all selected columns are removed,
        # it's an invalid statement. We return the node but it will be empty.
        if isinstance(node, exp.Select) and not node.expressions:
            print(f"--> All columns removed from a SELECT clause.")

        # If no rules match, keep the node.
        return node

    def prune(self):
        """
        Runs the pruning process on the AST.
        
        Returns:
            The modified AST object.
        """
        if not self.decorated_ast:
            return None
        
        # ast.transform() will recursively apply our transformer function
        pruned_ast = self.decorated_ast.transform(self._pruning_transformer)
        return pruned_ast

    def get_node_by_id(self, node_id):
        """Helper to find a node by its ID."""
        return self.decorated_ast.find(exp.Expression, finder=lambda n: hasattr(n, 'meta') and n.meta.get('id') == node_id)

    def pretty_print(self, ast_node=None):
        """Prints the tree structure of the given AST or the instance's AST."""
        # A simplified printer for demonstration
        if ast_node is None:
            ast_node = self.decorated_ast
        for node in ast_node.walk():
            indent = "  " * node.depth
            node_id = node.meta.get('id', 'N/A')[:8]
            print(f"{indent}├── {node.__class__.__name__} ({node.sql()}) [ID: {node_id}]")

In [ ]:
class ASTOperator:
    def __init__(self, ast_object): 
        self.ast_object = ast_object
        self.ast_tree = None 

    def convert_ast_to_tree(self):
        self.ast_tree = self.build_tree(self.ast_object)
        self.id_to_node_map = {
            node.meta['id']: node for node in self.ast_object.walk() if hasattr(node, 'meta')
        }

    def build_tree(self, sqlglot_node, parent=None):

        if parent is not None:
            clause_root = TreeNode(sqlglot_node, parent=parent)
            root_node = parent
            root_node.add_child(clause_root)
        else:
            root_node = TreeNode(sqlglot_node)
            clause_root = TreeNode(sqlglot_node, parent=root_node)
            root_node.add_child(clause_root)
        

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]

            for expr in children:
                if not isinstance(expr, exp.Expression): # did not understand the logic of the check
                    continue

                c_kind, c_name = map_node_type(expr)

                if not found_first_clause and c_kind != "Clause":
                    sub = self._build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=root_node)
                    root_node.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier): # this is a weak check, so try to transfer this is recursion
                                continue  # Skip Identifier under Table
                            child_node = self._build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue  # Skip Identifier under Table
                                if isinstance(item, exp.Expression):
                                    child_node = self._build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
        if parent is None:
            return root_node


    def _build_recursive(self, sqlglot_node, parent_node):
        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "ColumnRef":
            return None

        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "TableRef":
            return None
        
        if isinstance(sqlglot_node, exp.Identifier) and parent_node.name == "Alias":
            return None

        if isinstance(sqlglot_node, exp.TableAlias) and parent_node.name == "TableRef":
            return None

        if isinstance(sqlglot_node, exp.Paren):
            # return first child of the paren expression to avoid unnecessary nesting
            first_child = next(iter(sqlglot_node.args.values()))
            return self._build_recursive(first_child, parent_node)

        if isinstance(sqlglot_node, exp.Select):
            current = self.build_tree(sqlglot_node, parent_node)
        else:
            current = TreeNode(sqlglot_node, parent_node)
            for _, child in sqlglot_node.args.items():
                if isinstance(child, exp.Expression):
                    if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                        continue
                    child_node = self._build_recursive(child, current)
                    if child_node:
                        current.add_child(child_node)
                elif isinstance(child, list):
                    for item in child:
                        if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                            continue
                        if isinstance(item, exp.Expression):
                            child_node = self._build_recursive(item, current)
                            if child_node:
                                current.add_child(child_node)

        return current
    
    def get_node_children(self, node):
        """Helper to get all child expression nodes from a node's args."""
        for value in node.args.values():
            if isinstance(value, exp.Expression):
                yield value
            elif isinstance(value, list):
                for item in value:
                    if isinstance(item, exp.Expression):
                        yield item

    def pretty_print_ast(self, node, prefix="", is_last=True):
        """
        Recursively prints a formatted tree of the AST, including our custom IDs.
        """
        # Use the first 8 characters of the UUID for cleaner printing
        short_id = node.meta.get('id', 'N/A')[:8]
        
        # Get the node's type and any direct value (like a column name)
        node_name = node.__class__.__name__
        if hasattr(node, 'this'):
            node_name += f" ({node.this})"

        # Print the current node with a tree-like structure
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{node_name}  [ID: {short_id}]")

        # Recurse for children
        children = list(get_node_children(node))
        for i, child in enumerate(children):
            new_prefix = prefix + ("    " if is_last else "│   ")
            pretty_print_ast(child, prefix=new_prefix, is_last=(i == len(children) - 1))

    def remove_node_by_id(self, ast_node, target_id: str):
        """
        Removes a single node from the AST based on its unique 'meta.id'.

        This function uses ast.transform to walk the tree and delete the
        target node by returning None when it's found.

        Args:
            ast_node: The root node of the decorated sqlglot AST.
            target_id: The unique ID of the node to remove.

        Returns:
            A new AST with the specified node removed.
        """
        if not target_id:
            return ast_node

        # Define the transformer function inside this scope so it has access to 'target_id'.
        # This is a common pattern known as a closure.
        def transformer(node):
            # Check if the node has the ID we're looking for.
            if hasattr(node, 'meta') and node.meta.get('id') == target_id:
                # By returning None, we tell ast.transform to delete this node.
                # sqlglot correctly handles restructuring the parent.
                print(f"--> Found and removing node '{node.__class__.__name__}' with ID: {target_id}")
                return None
            
            # For all other nodes, return them unchanged to keep them in the tree.
            return node

        # Apply the configured transformer to the entire AST.
        return ast_node.transform(transformer)

    def replace_node_with_existing_node(self, ast_node, target_id: str, source_id: str):
        """
        Replaces a target node with another existing node from within the same AST.

        This is useful for "promotion" scenarios, where a child replaces its parent.

        Args:
            ast_node: The root node of the decorated sqlglot AST.
            target_id: The ID of the node to be replaced (the parent).
            source_id: The ID of the node that will take its place (the child).

        Returns:
            A new AST with the promotion completed.
        """

        source_node = self.id_to_node_map.get(source_id)
        if not source_node:
            print(f"[Warning] Could not find source node with ID: {source_id}")
            return ast_node

        # 2. The transformer now uses the pre-fetched source_node.
        def transformer(node):
            if hasattr(node, 'meta') and node.meta.get('id') == target_id:
                print(f"--> Promoting node [ID: {source_id}] to replace node [ID: {target_id}]")
                # Return the source_node, which we already found.
                # sqlglot's post-order traversal ensures this works correctly.
                return source_node
            return node

        return ast_node.transform(transformer)
    
    def add_condition_under_node_direct(self, ast_node, target_id: str, condition_sql: str):
        """
        Adds a WHERE condition to a target node, finding it directly via a helper.
        """
        # 1. "Directly" find the target node using our helper. The walk is hidden inside.
        target_node = self.id_to_node_map.get(target_id)
                
        if not target_node:
            print(f"[Warning] Could not find a node with ID: {target_id}")
            return ast_node

        if not isinstance(target_node, (exp.Select, exp.Update, exp.Delete)):
            print(f"[Warning] Node of type '{target_node.__class__.__name__}' cannot have a WHERE clause.")
            return ast_node
            
        print(f"--> Found target node '{target_node.__class__.__name__}' [ID: {target_id}]. Adding condition...")
        
        # 2. The rest of the logic is the same.
        target_node.where(condition_sql, copy=False)
        
        return ast_node
    
    def recursive_ast_pruning(self, node, scope=None): 
        """ 
        Recursively prune the tree by setting `remove = True` on nodes that are marked for removal.
        """
        if node.kind == "Clause":
            scope = node.name if scope is None else scope

        all_res = True 
        for child in node.children:
            res = self.recursive_ast_pruning(child)
            all_res = all_res and res

            if scope == "SelectClause":
                # If we are in SelectClause and a child is removed, we should remove the parent
                if res: 
                    if node.name == "Function":
                        node.remove = True 
                        self.remove_node_by_id(self.ast_object, node.id)
                    return True 
                else: 
                    return False  
            elif scope == "WhereClause":
                # If we are in WhereClause and a child is removed, we should remove the parent
                if res and node.name == "Operator":
                    node.remove = True 
                    self.remove_node_by_id(self.ast_object, node.id)
                    return True 
                else: 
                    return False
        
        if node.kind == "Clause":
            if all_res:
                node.remove = True 
                self.remove_node_by_id(self.ast_object, node.id)
                return True
            else:
                return False 
        
        
        if node.name in ["ColumnRef", "TableRef"]:
            state = column_ref_dict.get(node.id, "Aligned")
            if state == "Violated":
                node.remove = True 
                self.remove_node_by_id(self.ast_object, node.id)
                if isinstance(self.ast_object, exp.Select):
                    if scope == "SelectClause":
                        if node.parent.kind == "Expression":
                            return True 
                        else:
                            return False
                    elif scope == "WhereClause":
                        if node.parent.name == "Operator":
                            return True 
                        else:
                            return False 
                    elif scope == "FromClause":
                        return True 
                    elif scope == "GroupByClause": 
                        return True
                    elif scope == "OrderByClause":
                        return True


In [ ]:
def remove_node_by_id(ast_node, target_id: str):
    """
    Removes a single node from the AST based on its unique 'meta.id'.

    This function uses ast.transform to walk the tree and delete the
    target node by returning None when it's found.

    Args:
        ast_node: The root node of the decorated sqlglot AST.
        target_id: The unique ID of the node to remove.

    Returns:
        A new AST with the specified node removed.
    """
    if not target_id:
        return ast_node

    # Define the transformer function inside this scope so it has access to 'target_id'.
    # This is a common pattern known as a closure.
    def transformer(node):
        # Check if the node has the ID we're looking for.
        if hasattr(node, 'meta') and node.meta.get('id') == target_id:
            # By returning None, we tell ast.transform to delete this node.
            # sqlglot correctly handles restructuring the parent.
            print(f"--> Found and removing node '{node.__class__.__name__}' with ID: {target_id}")
            return None
        
        # For all other nodes, return them unchanged to keep them in the tree.
        return node

    # Apply the configured transformer to the entire AST.
    return ast_node.transform(transformer)

In [ ]:
def replace_node_with_existing_node(ast_node, target_id: str, source_id: str):
    """
    Replaces a target node with another existing node from within the same AST.

    This is useful for "promotion" scenarios, where a child replaces its parent.

    Args:
        ast_node: The root node of the decorated sqlglot AST.
        target_id: The ID of the node to be replaced (the parent).
        source_id: The ID of the node that will take its place (the child).

    Returns:
        A new AST with the promotion completed.
    """
    # 1. First, create a map of all IDs to their nodes for quick lookup.
    id_to_node_map = {
        node.meta['id']: node for node in ast_node.walk() if hasattr(node, 'meta')
    }

    source_node = id_to_node_map.get(source_id)
    if not source_node:
        print(f"[Warning] Could not find source node with ID: {source_id}")
        return ast_node

    # 2. The transformer now uses the pre-fetched source_node.
    def transformer(node):
        if hasattr(node, 'meta') and node.meta.get('id') == target_id:
            print(f"--> Promoting node [ID: {source_id}] to replace node [ID: {target_id}]")
            # Return the source_node, which we already found.
            # sqlglot's post-order traversal ensures this works correctly.
            return source_node
        return node

    return ast_node.transform(transformer)

In [ ]:
def add_condition_under_node_direct(ast_node, target_id: str, condition_sql: str):
    """
    Adds a WHERE condition to a target node, finding it directly via a helper.
    """
    # 1. "Directly" find the target node using our helper. The walk is hidden inside.
    target_node = find_node_by_id(ast_node, target_id)
            
    if not target_node:
        print(f"[Warning] Could not find a node with ID: {target_id}")
        return ast_node

    if not isinstance(target_node, (exp.Select, exp.Update, exp.Delete)):
        print(f"[Warning] Node of type '{target_node.__class__.__name__}' cannot have a WHERE clause.")
        return ast_node
        
    print(f"--> Found target node '{target_node.__class__.__name__}' [ID: {target_id}]. Adding condition...")
    
    # 2. The rest of the logic is the same.
    target_node.where(condition_sql, copy=False)
    
    return ast_node

### AST to sql

In [19]:
def remove_column(ast, node_id):
    #XXX: Logic seems to be complete but needs to be tested
    select_expressions = ast.expressions
    ast.set(
        "expressions",
        [expr for expr in select_expressions if not (hasattr(expr, 'id') and expr.id == node_id)]
    )


In [20]:
def apply_pruning_to_ast(tree_root, ast):
    """
    Traverse TreeNode tree and remove corresponding sqlglot AST nodes for nodes marked `.remove = True`.
    """
    for node in tree_root.walk():
        if getattr(node, "remove", False):
            # Remove corresponding sqlglot node from AST if it's a ColumnRef under SELECT
            if node.name == "ColumnRef":
                remove_column(ast, node.id)
            # TODO: Handle the ColumnRef that are not directly under the (SELECT or WHERE) clause, because most of the node under the WHERE clause would be under a Operator node or multiple Operator nodes.add()
            # But Also, I think this is somewhat handled by the soft_prune_ast function above

        # TODO: Need to handle the case where a TableRef node is marked for removal, which would require removing the entire statement or clause it belongs to.

In [21]:
apply_pruning_to_ast(tree_root, ast)

In [22]:
ast.sql()

'SELECT account_id, account_to, bank_to FROM Order'